# 01 — Phase 1: PDF Download + YOLO Crop Extraction
Downloads each PDF directly into RAM, runs DocLayout-YOLO, saves figure crops.
PDFs are **not saved to disk** — only the crops are written.

**Before running:** check config.yaml → subset.mode and n_first.

In [1]:
# ── CELL 1: Setup ────────────────────────────────────────────────
import sys; sys.path.insert(0, "..")
from src.config_loader import load_config
from src.patents import load_patents, get_subset
from src.extractor import extract_crops_streaming

cfg = load_config()

print("Config loaded.")
print(f"  Subset mode : {cfg['subset']['mode']}")
print(f"  n_first     : {cfg['subset']['n_first']}")
print(f"  YOLO device : {cfg['extractor']['device']}")
print(f"  PDF DPI     : {cfg['extractor']['pdf_dpi']}")
print(f"  Crops saved : {cfg['paths']['crops']}")

Config loaded.
  Subset mode : selected
  n_first     : 10
  YOLO device : cuda:1
  PDF DPI     : 150
  Crops saved : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/data/1639_Patents_Raw_Crops


---
**Optional:** If you ran  and saved a list, load it here instead of using the config subset mode.

In [2]:
# ── CELL 2: Load patents ──────────────────────────────────────────
# Option A: use config.yaml subset settings (default)
df, missing_df = load_patents(cfg)   # missing_df = patents with no PDF URL
subset         = get_subset(df, cfg)

# Option B: load from saved filter list (uncomment to use)
# from pathlib import Path
# ids = Path("../selected_patents.txt").read_text().strip().splitlines()
# subset = df[df["Record Number"].isin(ids)].copy()
# print(f"Loaded {len(subset)} patents from saved filter")

print(f"Ready to process {len(subset)} patents.")
subset[["Record Number", "Title"]].head(5)

Loading Excel: /mnt/storage_11tb/Drive_files_to_syncronize/2 - Patente & Validation/3 -Raw_Patent_Exports_PatSeer_&Gold_Standard/1639__dataset_08_06_26.xlsx


/home/vasco/anaconda3/envs/nb_00b2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/vasco/anaconda3/envs/nb_00b2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  11 patents have no PDF URL (will appear in status report)
  Loaded 1628 patents with valid PDF URLs.
  Subset mode='selected': 126 patents selected.
Ready to process 126 patents.


,Record Number,Title
0,US2021403155A1,VTOL AIRCRAFT
1,WO2024013392A1,VERTICAL TAKE-OFF AIRCRAFT
2,US2016244158A1,VERTICAL TAKE-OFF AND LANDING VEHICLE WITH INC...
3,US2024359791A1,AIRCRAFT
4,JP2024519073A,Electrical fault isolation in aircraft power d...


In [3]:
# ── CELL 3: Download + Extract (streaming — no PDFs saved) ──────────
# Downloads each PDF into RAM, runs YOLO, saves crops. No PDF files written.
# Already-processed patents are  skipped  automatically.
# missing_df passed so no-URL patents also appear in extraction_status.xlsx.

crop_results = extract_crops_streaming(subset, cfg, no_url_df=missing_df)

# Quick summary
total_crops = sum(len(v) for v in crop_results.values())
empty       = [k for k, v in crop_results.items() if len(v) == 0]
print(f"Total crops: {total_crops}")
print(f"Patents with 0 crops: {len(empty)}")
if empty:
    print("  ", empty[:10])

Download + Extract: 100%|██████████| 126/126 [18:54<00:00,  9.01s/it]

Status report: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/data/1639_Patents_logs/extraction_status.xlsx
Extraction complete: 1928 crops saved to /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/data/1639_Patents_Raw_Crops
Total crops: 1928
Patents with 0 crops: 1
   ['DE202024104658U1']


In [4]:
# ── CELL 4: Save crop index for Phase 2 ─────────────────────────────
# Saves a JSON map of { record_id: [list of crop file paths] }
# Phase 2 (02_processing.ipynb) reads this file to know what to process.

import json
from pathlib import Path

crop_index  = {k: [str(p) for p in v] for k, v in crop_results.items()}
out         = Path(cfg["paths"]["logs"]) / "crop_index.json"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w") as f:
    json.dump(crop_index, f, indent=2)
print(f"Crop index saved to: {out}")

Crop index saved to: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/data/1639_Patents_logs/crop_index.json
